# M2.S1 — Parallel Work & Decomposition

## Learning goals

By the end of this notebook you should be able to:

- identify independent work and dependencies;
- distinguish task decomposition from data decomposition;
- explain load imbalance and makespan;
- explain why task granularity matters;
- decompose a 2-D grid across workers;
- connect independent tasks to a SLURM job array.

> **Key idea:** Choose the decomposition before choosing OpenMP, MPI, CUDA, or another programming model.


## Exercise 1 — What can actually run in parallel?

Consider three problems:

**A.** Process 16 independent satellite images  
**B.** Compute a sequence where `x[i]` depends on `x[i-1]`  
**C.** Update an 8×8 climate grid where each cell depends on neighbouring cells

For each case identify:

1. the units of work;
2. which work is independent;
3. the dependencies;
4. whether task decomposition or data decomposition is more natural;
5. whether adding more workers will always help.


## Exercise 2 — Independent tasks

We have 16 independent tasks. The code below shows how many **waves** are needed for a given number of workers.


In [ ]:
tasks = list(range(16))

for workers in [4, 8, 16, 32]:
    print(f"\nWorkers = {workers}")
    for wave in range(0, len(tasks), workers):
        running = tasks[wave:wave + workers]
        print(f"  Wave {wave // workers + 1}: tasks {running}")


**Questions**

- How many waves are required with 4, 8, 16 and 32 workers?
- Why does moving from 16 to 32 workers not reduce the number of waves?
- What does this tell you about the maximum useful parallelism of this workload?


## Exercise 3 — Load balancing

Eight tasks have different durations:

`[8, 7, 6, 5, 4, 3, 2, 1]`

First try a poor allocation.


In [ ]:
tasks = [8, 7, 6, 5, 4, 3, 2, 1]

poor_allocation = {
    "Worker 1": [8, 7],
    "Worker 2": [6, 5],
    "Worker 3": [4, 3],
    "Worker 4": [2, 1],
}

for worker, work in poor_allocation.items():
    print(worker, work, "total =", sum(work))

poor_makespan = max(sum(work) for work in poor_allocation.values())
print("Parallel execution time (makespan):", poor_makespan)


Now compare it with a balanced allocation.


In [ ]:
balanced_allocation = {
    "Worker 1": [8, 1],
    "Worker 2": [7, 2],
    "Worker 3": [6, 3],
    "Worker 4": [5, 4],
}

for worker, work in balanced_allocation.items():
    print(worker, work, "total =", sum(work))

balanced_makespan = max(sum(work) for work in balanced_allocation.values())
print("Parallel execution time (makespan):", balanced_makespan)


> **Takeaway:** Parallel execution finishes when the slowest worker finishes.

**Question:** Can you find another allocation with the same optimal makespan?


## Exercise 4 — Granularity

Breaking work into more tasks can improve scheduling flexibility, but every task also has overhead.

The model below keeps the useful work fixed and increases the number of tasks.


In [ ]:
total_useful_work = 10.0      # seconds
overhead_per_task = 0.002     # seconds

print(" tasks | overhead(s) | total(s)")
print("--------------------------------")

for number_of_tasks in [1, 10, 100, 1000, 10000]:
    overhead = number_of_tasks * overhead_per_task
    total = total_useful_work + overhead
    print(f"{number_of_tasks:6d} | {overhead:11.3f} | {total:8.3f}")


**Questions**

- Why are tasks that are too large a problem?
- Why are tasks that are too small a problem?
- What does a useful granularity try to balance?

> **Takeaway:** Enough tasks to keep workers busy — but not so many tiny tasks that overhead dominates.


## Exercise 5 — Data decomposition of a 2-D grid

We divide an 8×8 grid among four workers.


In [ ]:
import numpy as np

grid = np.arange(64).reshape(8, 8)
grid


In [ ]:
workers = {
    "W0": grid[:4, :4],
    "W1": grid[:4, 4:],
    "W2": grid[4:, :4],
    "W3": grid[4:, 4:]
}

for worker, block in workers.items():
    print(f"\n{worker}")
    print(block)


Now assume that every cell needs information from its **north, south, east and west neighbours**.

**Questions**

- Which worker pairs need to exchange boundary information?
- Which cells lie on those boundaries?
- Why is this still parallel, even though communication is required?

These neighbouring boundary values later become **halo/ghost cells** in distributed-memory programs such as MPI.


## Exercise 6 — Dependencies and a task graph

Consider:

```text
A ──► C ──► E
 \         ▲
  ─► D ───┘

B ──► D
```

| Task | Duration | Depends on |
|---|---:|---|
| A | 3 | — |
| B | 2 | — |
| C | 4 | A |
| D | 3 | A, B |
| E | 2 | C, D |

**Questions**

1. Which tasks can start at time 0?
2. When can C start?
3. When can D start?
4. What determines when E can start?
5. Which dependency chain forms the critical path?


## Exercise 7 — From decomposition to SLURM

The accompanying files in `scripts/m2s1/` demonstrate the same idea on a real cluster:

- `process_task.py`
- `tasks_array.slurm`

The SLURM job array launches 16 independent tasks:

```text
16 independent tasks
        ↓
SLURM job array
        ↓
scheduler
        ↓
compute nodes
```

From the `scripts/m2s1/` folder run:

```bash
sbatch tasks_array.slurm
squeue -u $USER
```

After the job completes:

```bash
cat task_*.out
```

Look at the hostnames. Depending on the scheduler and available resources, tasks may run on different compute nodes.


## Final challenge — Mandelbrot and load imbalance

Imagine splitting a Mandelbrot image into equal horizontal stripes.

**Question:** If all stripes contain the same number of pixels, will all workers necessarily finish at the same time?

Why or why not?

Think about:

- equal data size versus equal computational work;
- static versus dynamic scheduling;
- how decomposition affects load balance.

---

## Final takeaway

> **Before choosing OpenMP, MPI or CUDA, identify the parallel structure of the problem: independent work, dependencies, communication, granularity and load balance.**
